In [ ]:
import torch
from datasets import load_dataset
import numpy as np
import pandas as pd
from allennlp.modules.scalar_mix import ScalarMix
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import classification_report, f1_score
from sklearn.utils import shuffle
from tqdm import tqdm
import transformers

In [ ]:
device = torch.device('cuda')

#Original dataset takes too long to load because of images
ds = load_dataset("tasksource/ScienceQA_text_only")
label_names = ['elementary', 'middle', 'high']

def grade_to_label(grade):
    if isinstance(grade, str):
        num = int(''.join(filter(str.isdigit, grade)))
    else:
        num = int(grade)
    if 1 <= num <= 5:
        return 0  # elementary
    elif 6 <= num <= 8:
        return 1  # middle
    elif 9 <= num <= 12:
        return 2  # high
    return None

def prepare_scienceqa(split_data):
    texts = []
    labels = []
    for row in split_data:
        label = grade_to_label(row.get('grade'))
        if label is None:
            continue

        question = row['question']
        choices = row['choices']
        answer_idx = row['answer']
        answer_text = choices[answer_idx] if answer_idx < len(choices) else ""
        lecture = row.get('lecture', '') or ''
        solution = row.get('solution', '') or ''

        text = f"""Question: {question}
        Choices: {', '.join(f'{chr(65+i)}) {c}' for i, c in enumerate(choices))}
        Correct Answer: {chr(65+answer_idx)}) {answer_text}
        Explanation: {lecture}
        Solution: {solution}"""

        texts.append(text)
        labels.append(label)
    return np.array(texts), np.array(labels)

X_train, y_train = prepare_scienceqa(ds['train'])
X_test, y_test = prepare_scienceqa(ds['test'])

for i in range(10):
    print(f"Label: {y_train[i]} | Text: {X_train[i]}")
    print("\n")


In [ ]:
samples_per_class = 1516
balanced_index = []
for label in range(3):
    index = np.where(y_train == label)[0]
    n = min(len(index), samples_per_class)
    balanced_index.extend(np.random.choice(index, n, replace=False))
np.random.shuffle(balanced_index)
X_train = X_train[balanced_index]
y_train = y_train[balanced_index]

In [ ]:
import torch
import torch.nn as nn


_COEFFS = {
    "leaky_relu": (
        [0.029792778657264946, 0.6183735264987601, 2.323309062531321,
         3.051936237265109, 1.4854203263828845, 0.2510244961111299],
        [-1.1419548357285474, 4.393159974992486, 0.8714712309957245,
         0.34719662339598834],
    ),
    "relu": (
        [0.029963801610813613, 0.6168978366891341, 2.37534759733888,
         3.0659900472408443, 1.5246831881677423, 0.2528070864040542],
        [-1.191550121923625, 4.4080487697236626, 0.9110357113686055,
         0.34884977946384615],
    ),
    "gelu": (
        [-0.0012423594497499122, 0.5080497063245629, 0.41586363182937475,
         0.13022718688035761, 0.024355900098993424, 0.00290283948155535],
        [-0.06675015696494944, 0.17927646217001553, 0.03746682605496631,
         1.6561610853276082e-10],
    ),
    "swish": (
        [3.054879741161051e-07, 0.5000007853744493, 0.24999783422824703,
         0.05326628273219478, 0.005803034571292244, 0.0002751961022402342],
        [-4.111554955950634e-06, 0.10652899335007572, -1.2690007399796238e-06,
         0.0005502331264140556],
    ),
    "tanh": (
        [-1.0804622559204184e-08, 1.0003008043819048, -2.5878199375289335e-08,
         0.09632129918392647, 3.4775841628196104e-09, 0.0004255709234726337],
        [-0.0013027181209176277, 0.428349017422072, 1.4524304083061898e-09,
         0.010796648111337176],
    ),
    "sigmoid": (
        [0.4999992534599381, 0.25002157564685185, 0.14061924500301096,
         0.049420492431596394, 0.00876714851885483, 0.0006442412789159799],
        [2.1694506382753683e-09, 0.28122766100417684, 1.0123620714203357e-05,
         0.017531988049946],
    ),
}


class Rational(nn.Module):
    def __init__(self, approx_func="leaky_relu"):
        super().__init__()
        num, den = _COEFFS[approx_func]
        self.numerator = nn.Parameter(torch.tensor(num, dtype=torch.float32))
        self.denominator = nn.Parameter(torch.tensor(den, dtype=torch.float32))

    def forward(self, x):
        P = sum(self.numerator[i] * x**i for i in range(6))
        Q = sum(self.denominator[j] * x**(j+1) for j in range(4))
        return P / (1 + Q.abs())


class ScoringModel(torch.nn.Module):
    def __init__(self, language_model, prefix, num_classes=3, approx_func="leaky_relu") -> None:
        super().__init__()
        self.prefix = prefix

        self.tokenizer = AutoTokenizer.from_pretrained(language_model)
        self.lm = AutoModel.from_pretrained(language_model).to(device)
        self.scalar_mix = ScalarMix(self.lm.config.num_hidden_layers + 1)

        self.dropout = torch.nn.Dropout(p=0.2)

        self.lm_name = language_model

        self.classification_head = torch.nn.Sequential(
            torch.nn.Linear(self.lm.config.hidden_size, self.lm.config.hidden_size),
            Rational(approx_func=approx_func),
            torch.nn.Linear(self.lm.config.hidden_size, num_classes)
        )

        self.loss = torch.nn.CrossEntropyLoss()

        self.X = None
        self.y = None
        self.eval_X = None
        self.eval_y = None

        # If set, every epoch summary / eval report is appended to this file
        # in addition to being printed to stdout.
        self.log_path = None

    def set_dataset(self, X, y):
        self.X = X
        self.y = y

    def set_evalset(self, X, y):
        self.eval_X = X
        self.eval_y = y

    def _log(self, msg=""):
        """Print to stdout and append to self.log_path (if set)."""
        print(msg)
        if self.log_path is not None:
            with open(self.log_path, "a") as f:
                f.write(str(msg) + "\n")

    def self_eval(self):
        self.eval()
        predictions = []

        with torch.no_grad():
            for text in tqdm(self.eval_X):
                logits = self.forward(text)
                predicted_class = logits.argmax(dim=1).item()
                predictions.append(predicted_class)

        true_labels = self.eval_y
        if torch.is_tensor(true_labels):
            true_labels = true_labels.cpu().numpy()

        report = classification_report(
            true_labels, predictions,
            target_names=['elementary', 'middle', 'high'],
        )
        self._log(report)

        return {'macro_f1': f1_score(true_labels, predictions, average='macro')}

    def forward(self, input):
        inputs = self.tokenizer(input, return_tensors='pt', padding=True, truncation=True, max_length=self.lm.config.max_position_embeddings - 2)
        outputs = self.lm(**inputs.to(device), output_hidden_states=True)
        hidden_states = outputs.hidden_states

        result = self.classification_head(
            torch.mean(
                self.dropout(
                    self.scalar_mix(
                        hidden_states
                    )
                ),
                dim=1
            )
        )
        return result

    def fit(
        self,
        epochs,
        optimizer,
        scheduler,
        batch_size=4
    ) -> None:
        self.train()
        for epoch in range(epochs):
            self._log(f"Epoch {epoch}")
            r = 0.0
            num_s = 0.0
            d, d_y = shuffle(self.X, self.y)

            batches_X = [
                d[n:n+batch_size] for n in range(0, len(d), batch_size)
            ]
            batches_y = [
                d_y[n:n+batch_size] for n in range(0, len(d_y), batch_size)
            ]

            for batch in tqdm(range(len(batches_X))):
                pred = self.forward(list(batches_X[batch]))
                ls = self.loss(pred, batches_y[batch])

                optimizer.zero_grad()
                ls.backward()
                optimizer.step()
                scheduler.step()

                r += ls.detach().item()
                num_s += 1

                if batch % 10 == 0 and batch > 0:
                    print(str(r / num_s))

            self._log(f"  Epoch {epoch}: avg loss {r / max(num_s, 1):.4f}")

            if not (self.eval_X is None):
                ev = self.self_eval()
                self._log(ev)
                self.train()


In [ ]:
import os

y_train_tensor = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)

approx_funcs = ["leaky_relu", "relu", "gelu", "swish", "tanh", "sigmoid"]
results = {}

# Directory for per-activation log files (and the final summary).
LOG_DIR = 'rational_logs'
os.makedirs(LOG_DIR, exist_ok=True)

for approx in approx_funcs:
    model = ScoringModel(
        'google/electra-large-discriminator',
        prefix=f'run-{approx}',
        num_classes=3,
        approx_func=approx
    ).to(device)
    model.set_dataset(X_train, y_train_tensor)
    model.set_evalset(X_test, y_test_tensor)

    # Fresh log file for this activation.
    log_path = os.path.join(LOG_DIR, f'training_log_{approx}.txt')
    open(log_path, 'w').close()  # truncate any prior log
    model.log_path = log_path

    model._log(f'=== Activation: {approx} ===')

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
    scheduler = transformers.get_cosine_schedule_with_warmup(
        optimizer=optimizer, num_warmup_steps=50,
        num_training_steps=3 * len(X_train))

    model.fit(5, optimizer, scheduler=scheduler, batch_size=8)

    model._log('--- Final evaluation ---')
    metrics = model.self_eval()
    model._log(metrics)
    results[approx] = metrics['macro_f1']

    torch.save(model.state_dict(), f'final-model-electra-scienceqa-{approx}.pt')

    del model, optimizer, scheduler
    torch.cuda.empty_cache()

# Final summary: print AND write to a summary file.
summary_path = os.path.join(LOG_DIR, 'summary_f1_scores.txt')
with open(summary_path, 'w') as f:
    header = "Summary of F1 Scores"
    print(header); f.write(header + "\n")
    for k, v in sorted(results.items(), key=lambda kv: -kv[1]):
        line = f"{k:>11}: {v:.4f}"
        print(line); f.write(line + "\n")

print(f'\n[Per-activation logs in: {os.path.abspath(LOG_DIR)}]')
print(f'[Summary written to: {os.path.abspath(summary_path)}]')
